# Phase 3: Merge Clinical Predictors

This notebook loads the calibrated Phase 2 cohort, downloads public NHANES predictor files for 1999–2000 and 2001–2002, merges them by `SEQN`, creates a restrained set of pre–cystatin C predictors, summarizes missingness, and exports the final pre-modeling dataset.

Upload `Cystatin_C_Phase2_Calibrated_Outputs.zip` when prompted.

In [ ]:
import zipfile, warnings, requests
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)

ROOT = Path("/content/cystatin_c_phase3")
RAW = ROOT / "raw"
OUT = ROOT / "outputs"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("✅ Environment ready")

In [ ]:
# Upload Phase 2 ZIP
from google.colab import files
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one ZIP: Cystatin_C_Phase2_Calibrated_Outputs.zip")

phase2_zip = ROOT / zip_names[0]
phase2_zip.write_bytes(uploaded[zip_names[0]])

with zipfile.ZipFile(phase2_zip) as z:
    z.extractall(ROOT)

base_path = ROOT / "nhanes_calibrated_analytic.csv"
if not base_path.exists():
    raise FileNotFoundError("nhanes_calibrated_analytic.csv was not found in the ZIP.")

base = pd.read_csv(base_path)
base["SEQN"] = pd.to_numeric(base["SEQN"], errors="coerce").astype("Int64")
print("✅ Base cohort:", base.shape)

In [ ]:
# Official NHANES files
URLS = {
    "bmx_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/BMX.XPT",
    "bmx_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/BMX_B.XPT",
    "bpx_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/BPX.XPT",
    "bpx_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/BPX_B.XPT",
    "cbc_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/LAB25.XPT",
    "cbc_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L25_B.XPT",
    "urine_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/LAB16.XPT",
    "urine_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L16_B.XPT",
    "crp_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/LAB11.XPT",
    "crp_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L11_B.XPT",
    "diq_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/DIQ.XPT",
    "diq_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/DIQ_B.XPT",
    "bpq_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/BPQ.XPT",
    "bpq_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/BPQ_B.XPT",
    "smq_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/SMQ.XPT",
    "smq_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/SMQ_B.XPT",
    "mcq_1999": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1999/DataFiles/MCQ.XPT",
    "mcq_2001": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/MCQ_B.XPT",
}

def download(name, url):
    path = RAW / f"{name}.XPT"
    if path.exists():
        return path
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    if b"<html" in r.content[:500].lower() or len(r.content) < 1000:
        raise RuntimeError(f"Invalid download for {name}: {url}")
    path.write_bytes(r.content)
    print(f"✅ {name}: {path.stat().st_size/1024:.1f} KB")
    return path

paths = {k: download(k,v) for k,v in URLS.items()}

In [ ]:
def read_xpt(path):
    df = pd.read_sas(path, format="xport", encoding="utf-8")
    df.columns = df.columns.str.upper().str.strip()
    df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce").astype("Int64")
    return df

raw = {k: read_xpt(v) for k,v in paths.items()}
for k,v in raw.items():
    print(k, v.shape)

In [ ]:
# Combine each component across cycles before merging.
component_pairs = {
    "BMX": ("bmx_1999", "bmx_2001"),
    "BPX": ("bpx_1999", "bpx_2001"),
    "CBC": ("cbc_1999", "cbc_2001"),
    "URINE": ("urine_1999", "urine_2001"),
    "CRP": ("crp_1999", "crp_2001"),
    "DIQ": ("diq_1999", "diq_2001"),
    "BPQ": ("bpq_1999", "bpq_2001"),
    "SMQ": ("smq_1999", "smq_2001"),
    "MCQ": ("mcq_1999", "mcq_2001"),
}

components = {}
for component, (a,b) in component_pairs.items():
    temp = pd.concat([raw[a], raw[b]], ignore_index=True, sort=False)
    temp = temp.drop_duplicates("SEQN")
    components[component] = temp
    print(component, temp.shape)

merged = base.copy()
for component, data in components.items():
    # Prevent duplicate names already found in the calibrated base.
    add_cols = ["SEQN"] + [c for c in data.columns if c != "SEQN" and c not in merged.columns]
    merged = merged.merge(data[add_cols], on="SEQN", how="left", validate="one_to_one")

print("✅ Fully merged shape:", merged.shape)

In [ ]:
# Helper: return the first existing variable among cycle-compatible candidates.
def first_existing(data, candidates, default=np.nan):
    for c in candidates:
        if c in data.columns:
            return pd.to_numeric(data[c], errors="coerce")
    return pd.Series(default, index=data.index, dtype="float64")

# Anthropometrics
merged["BMI"] = first_existing(merged, ["BMXBMI"])
merged["WAIST_CM"] = first_existing(merged, ["BMXWAIST"])

# Mean systolic and diastolic BP from available readings.
sbp_cols = [c for c in ["BPXSY1","BPXSY2","BPXSY3","BPXSY4"] if c in merged.columns]
dbp_cols = [c for c in ["BPXDI1","BPXDI2","BPXDI3","BPXDI4"] if c in merged.columns]
merged["MEAN_SBP"] = merged[sbp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
merged["MEAN_DBP"] = merged[dbp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Laboratory predictors
merged["HEMOGLOBIN"] = first_existing(merged, ["LBXHGB"])
merged["CRP_MG_DL"] = first_existing(merged, ["LBXCRP"])
merged["URINE_ALBUMIN"] = first_existing(merged, ["URXUMA"])
merged["URINE_CREATININE"] = first_existing(merged, ["URXUCR"])

# UACR: urine albumin mg/L divided by urine creatinine mg/dL, multiplied by 100.
merged["UACR_MG_G"] = (
    merged["URINE_ALBUMIN"] / merged["URINE_CREATININE"] * 100
)

# Questionnaire indicators. NHANES generally codes Yes=1, No=2.
diabetes_raw = first_existing(merged, ["DIQ010"])
hypertension_raw = first_existing(merged, ["BPQ020"])
smoked_100_raw = first_existing(merged, ["SMQ020"])
current_smoke_raw = first_existing(merged, ["SMQ040"])

merged["DIAGNOSED_DIABETES"] = np.where(
    diabetes_raw.eq(1), 1,
    np.where(diabetes_raw.eq(2), 0, np.nan)
)
merged["DIAGNOSED_HYPERTENSION"] = np.where(
    hypertension_raw.eq(1), 1,
    np.where(hypertension_raw.eq(2), 0, np.nan)
)

# Smoking: 0 never, 1 former, 2 current.
merged["SMOKING_STATUS"] = np.select(
    [
        smoked_100_raw.eq(2),
        smoked_100_raw.eq(1) & current_smoke_raw.isin([1,2]),
        smoked_100_raw.eq(1) & current_smoke_raw.eq(3),
    ],
    [0, 2, 1],
    default=np.nan
)

# Cardiovascular disease: CHF, coronary heart disease, angina, heart attack, or stroke.
cvd_candidates = ["MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F"]
cvd_cols = [c for c in cvd_candidates if c in merged.columns]
if cvd_cols:
    cvd_raw = merged[cvd_cols].apply(pd.to_numeric, errors="coerce")
    merged["ANY_CVD"] = np.where(
        cvd_raw.eq(1).any(axis=1), 1,
        np.where(cvd_raw.eq(2).all(axis=1), 0, np.nan)
    )
else:
    merged["ANY_CVD"] = np.nan

print("✅ Core predictors derived")

In [ ]:
# Restrained predictor set for the future model.
predictors = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "CREATININE_CALIBRATED",
    "LBXSBU",
    "LBXSAL",
    "LBXSGL",
    "BMI",
    "WAIST_CM",
    "MEAN_SBP",
    "MEAN_DBP",
    "HEMOGLOBIN",
    "CRP_MG_DL",
    "UACR_MG_G",
    "DIAGNOSED_DIABETES",
    "DIAGNOSED_HYPERTENSION",
    "SMOKING_STATUS",
    "ANY_CVD",
]

outcomes = [
    "DISCORDANCE_30_CALIBRATED",
    "HIDDEN_CKD_CALIBRATED",
]

available_predictors = [c for c in predictors if c in merged.columns]
model_df = merged[
    ["SEQN","CYCLE","WTSCY4YR","SDMVPSU","SDMVSTRA"]
    + available_predictors + outcomes
].copy()

missingness = pd.DataFrame({
    "Variable": available_predictors,
    "Nonmissing N": [model_df[c].notna().sum() for c in available_predictors],
    "Missing N": [model_df[c].isna().sum() for c in available_predictors],
    "Missing (%)": [100*model_df[c].isna().mean() for c in available_predictors],
}).sort_values("Missing (%)", ascending=False)

display(missingness.round(2))
print("Primary outcome cases:", int(model_df["DISCORDANCE_30_CALIBRATED"].sum()))
print("Rows:", len(model_df))

In [ ]:
# Save outputs
model_path = OUT / "nhanes_final_predictor_dataset.csv"
missing_path = OUT / "predictor_missingness.csv"
merged_path = OUT / "nhanes_full_merged_predictors.csv"

model_df.to_csv(model_path, index=False)
missingness.to_csv(missing_path, index=False)
merged.to_csv(merged_path, index=False)

output_zip = Path("/content/Cystatin_C_Phase3_Predictor_Outputs.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in [model_path, missing_path, merged_path]:
        z.write(p, arcname=p.name)

print("✅ Created:", output_zip)
files.download(str(output_zip))

## After the notebook finishes

Upload `Cystatin_C_Phase3_Predictor_Outputs.zip`.

The next phase will decide which variables are usable based on missingness and then fit leakage-safe, cycle-aware models.